In [ ]:
# Cell 2: Environment Setup.
# ORIGINAL

import os
os.environ["fix_mistral_regex"] = "True"
# os.environ["OMP_NUM_THREADS"] = "1"
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # Extra 30% context lengths

# Install dependencies (run this if not already installed)
# !pip install unsloth vllm
# !pip install transformers==4.56.2
# !pip install --no-deps trl==0.22.2

In [ ]:
# Cell 3: Set remote HF_TOKEN from local .env
# ORIGINAL
import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv('HF_TOKEN')

# ssh -i ~/.ssh/id_ed25519 dataimaginations-heirarchical-reasoning@ssh.hf.space "echo 'export HF_TOKEN={hf_token}' >> ~/.bashrc"
print("✅ Token set! Restart remote shell to activate.")

In [ ]:
# Cell 4: HuggingFace Login
# ORIGINAL

import os
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print("✅ Logged in with HF_TOKEN")
else:
    login()
    print("✅ Logged in interactively")

In [ ]:
# Cell 5: Load Model
# ORIGINAL
from unsloth import FastLanguageModel
import torch

# Configuration
max_seq_length = 2048
lora_rank = 128      
lora_alpha = 128     # <--- Generally keep Alpha = Rank for Unsloth

print(f"⏳ Loading model with Rank {lora_rank}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3.5-mini-instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False,
)

print("🔗 Attaching LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,        # <--- FIX: Use the variable, don't hardcode!
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_alpha, # Set this to match the rank
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print(f"✅ Model loaded with Rank {lora_rank}!")

# New Block for MetaController

Uses LoRa instead of 4096 x 4096 Block

# --- Phase 1: The Metacontroller ("The Manager") ---

The paper describes a specific architecture (Appendix D.2 1) that generates "controllers" ($U_t$) to modify the residual stream.

In [ ]:
# Cell 6: Metacontroller
# TEMPORAL

import torch
import torch.nn as nn
import torch.nn.functional as F

class Metacontroller(nn.Module):
    def __init__(self, embed_dim=4096, latent_dim=16, hidden_dim=64):
        """
        The 'Manager' that lives inside the model.
        Args:
            embed_dim: Dimension of model's residual stream (e.g., 4096 for 8B)
            latent_dim: Size of the 'thought vector' (z)
        """
        super().__init__()
        self.latent_dim = latent_dim
        
        # 1. State Tracker (History) - Equation 12 [cite: 637]
        # Keeps track of what the model has been doing
        self.history_rnn = nn.GRUCell(embed_dim, hidden_dim)
        
        # 2. Policy Head (The Actor)
        # Decides on the next 'abstract action' (z) given history
        self.policy_mean = nn.Linear(hidden_dim, latent_dim)
        self.policy_logstd = nn.Linear(hidden_dim, latent_dim)
        
        # 3. Switching Unit (The Clock) - Equation 16 [cite: 644]
        # Decides: "Keep doing current thought" (beta=0) or "Switch to new thought" (beta=1)
        self.switch_head = nn.Sequential(
            nn.Linear(hidden_dim + latent_dim, 1),
            nn.Sigmoid()
        )
        
        # 4. Controller Decoder (Hypernetwork) - Equation 18 [cite: 649]
        # Converts the thought 'z' into actual interference vectors 'U'
        # We use Low-Rank (A and B) to save memory.
        self.rank = 8
        self.hyper_A = nn.Linear(latent_dim, embed_dim * self.rank)
        self.hyper_B = nn.Linear(latent_dim, embed_dim * self.rank)

    def forward(self, residual_input, prev_hidden, prev_z):
        """
        Runs one step of the Manager.
        """
        # Update history with current brain state of Llama
        hidden = self.history_rnn(residual_input, prev_hidden)
        
        # 1. Decide if we need a new plan (Switching)
        # Check based on history and PREVIOUS plan
        switch_prob = self.switch_head(torch.cat([hidden, prev_z], dim=-1))
        
        # 2. Generate a proposal for a new plan
        mu = self.policy_mean(hidden)
        std = torch.exp(self.policy_logstd(hidden))
        dist = torch.distributions.Normal(mu, std)
        proposal_z = dist.rsample() # Sampling with reparameterization
        
        # 3. Temporal Integration - Equation 2 [cite: 134]
        # If switch_prob is high, take new z. If low, keep ORIGINAL z.
        # For RL, we often binarize this (Algorithm 1)
        new_z = switch_prob * proposal_z + (1 - switch_prob) * prev_z
        
        # 4. Create the intervention (The "Steering")
        # Generate LoRA matrices A and B from z
        batch_size = residual_input.shape[0]
        matrix_A = self.hyper_A(new_z).view(batch_size, -1, self.rank)
        matrix_B = self.hyper_B(new_z).view(batch_size, self.rank, -1)
        
        # The control vector U*e = B @ A @ e
        # This is the "nudge" we apply to model's brain
        return new_z, hidden, switch_prob, matrix_A, matrix_B

# --- Phase 2: The Hook (Connecting Brains) ---

To make this work with Hugging Face, we use PyTorch Hooks. This intercepts the data flowing through layer 12 (or whichever layer you choose, paper suggests mid-depth ) and lets the Metacontroller modify it.

In [ ]:
# Cell 7: InternalRLWrapper
# TEMPORAL

class InternalRLWrapper:
    def __init__(self, base_model, metacontroller, target_layer=16):
        self.model = base_model
        self.meta = metacontroller
        self.target_layer = target_layer
        self.hook_handle = None
        
        # Storage for runtime states
        self.meta_hidden = None
        self.current_z = None
        self.intervention_A = None
        self.intervention_B = None
        
    def _hook_function(self, module, input, output):
        """
        The magic function that runs INSIDE model layer execution.
        Equation 11: e_hat = e + U*e [cite: 633]
        """
        # 'output' is the residual stream activation (batch, seq, dim)
        # We only intervene on the LAST token (during generation)
        current_activation = output[0][:, -1, :] 
        
        if self.meta_hidden is None:
            # Initialize state if first step
            batch_size = current_activation.shape[0]
            self.meta_hidden = torch.zeros(batch_size, 64, device=output[0].device)
            self.current_z = torch.zeros(batch_size, 16, device=output[0].device)

        # Run Metacontroller
        z, h, beta, A, B = self.meta(current_activation, self.meta_hidden, self.current_z)
        
        # Update states
        self.meta_hidden = h
        self.current_z = z
        
        # Apply Intervention (Low Rank Control)
        # delta = (e @ A) @ B
        # Reshaping for matrix multiplication
        act_unsqueezed = current_activation.unsqueeze(1) # (B, 1, Dim)
        delta = torch.bmm(torch.bmm(act_unsqueezed, A), B).squeeze(1)
        
        # Inject the thought!
        # Modify the output IN PLACE
        output[0][:, -1, :] = output[0][:, -1, :] + delta
        
        return output

    def register(self):
        # Attach to the specific layer of Llama
        layer = self.model.model.layers[self.target_layer]
        self.hook_handle = layer.register_forward_hook(self._hook_function)
        
    def remove(self):
        if self.hook_handle:
            self.hook_handle.remove()

# --- Phase 3: The Training Loop (Algorithms 1 & 3) ---

This corresponds to Algorithm 3  in the paper. It treats the Llama model + Metacontroller as an environment.

HICRA Integration: 

This is where you reward the Metacontroller if the model outputs "Strategic Grams".

# NOTE: I need to convert the newer HICRA setup from Cell 10 to this cell.

In [ ]:
# Cell 8: HICRA merged into the metacontroller code
# TEMPORAL (this needs to be adapted with the better HICRA from cell 10)
import torch.optim as optim

def train_internal_rl(wrapper, tokenizer, dataset, num_steps=1000):
    # Optimizer only trains the Metacontroller! Llama is frozen.
    optimizer = optim.AdamW(wrapper.meta.parameters(), lr=3e-5)
    
    # Enable the hook
    wrapper.register()
    
    for step in range(num_steps):
        # 1. Get Data
        prompt = dataset[step]['prompt']
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        # 2. Reset Internal State (Algorithm 2 [cite: 575])
        wrapper.meta_hidden = None 
        
        # 3. Generate with Intervention (Rollout)
        # The hook will automatically fire at every token step
        outputs = wrapper.model.generate(
            **inputs, 
            max_new_tokens=100,
            output_scores=True,
            return_dict_in_generate=True
        )
        
        generated_text = tokenizer.decode(outputs.sequences[0])
        
        # --- HICRA INTEGRATION HERE ---
        # Did the internal intervention cause the model to plan?
        hicra_score = 0.0
        for gram in ["let's assume", "first i need", "alternatively"]:
            if gram in generated_text.lower():
                hicra_score += 1.0 # Reward the Manager for making model smart
        
        # Check final answer correctness
        correctness_score = 1.0 if "correct_answer" in generated_text else 0.0
        
        total_reward = hicra_score + correctness_score
        
        # 4. Compute Loss (Simplified PPO/REINFORCE for Metacontroller)
        # In a full implementation, you'd save log_probs of 'z' and 'beta' 
        # inside the hook and use them here.
        # loss = -log_prob(z) * total_reward
        
        # optimizer.zero_grad()
        # loss.backward()
        # optimizer.step()
        
        print(f"Step {step} | Reward: {total_reward} | Text: {generated_text[:50]}...")

    wrapper.remove()

In [ ]:
# Cell 9: Load and Combine Datasets
from datasets import load_dataset, Dataset
import json

# === Configuration ===
MAX_PROMPT_TOKENS = 400    # Filter out prompts longer than this
MAX_ANSWER_TOKENS = 600    # Filter out answers longer than this  
NEMOTRON_SAMPLE_SIZE = 3000  # How many Nemotron examples to use

# System prompt for reasoning format
SYSTEM_PROMPT = """
You are a mathematical reasoning assistant. Think through problems step by step.
Respond in the following format:
<think>
...
</think>
<answer>
...
</answer>
"""

def format_prompt(example):
    """Format dataset for GRPO training with chat template."""
    return {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT.strip()},
            {'role': 'user', 'content': example['prompt']}
        ],
        'answer': str(example['answer'])
    }

def format_nemotron(example):
    """Convert Nemotron format to our format."""
    messages = example.get('messages', [])
    
    # Extract user prompt and assistant answer
    user_content = ""
    assistant_content = ""
    
    for msg in messages:
        if msg['role'] == 'user':
            user_content = msg['content']
        elif msg['role'] == 'assistant':
            assistant_content = msg['content']
    
    # Get expected answer (fallback to assistant content if not available)
    expected = example.get('expected_answer', '')
    if not expected:
        # Try to extract from assistant's <answer> tags if present
        if '<answer>' in assistant_content and '</answer>' in assistant_content:
            expected = assistant_content.split('<answer>')[-1].split('</answer>')[0].strip()
        else:
            expected = assistant_content[-200:] if len(assistant_content) > 200 else assistant_content
    
    return {
        'prompt': user_content,
        'answer': str(expected)
    }

def estimate_tokens(text):
    """Rough token estimate (1 token ≈ 4 chars for English)."""
    return len(str(text)) // 4

def filter_by_length(example):
    """Filter out examples that are too long."""
    prompt_tokens = estimate_tokens(example['prompt'])
    answer_tokens = estimate_tokens(example['answer'])
    return prompt_tokens <= MAX_PROMPT_TOKENS and answer_tokens <= MAX_ANSWER_TOKENS

# === 1. Load Your HICRA Synthetic Data ===
print("📂 Loading HICRA dataset...")
my_dataset = load_dataset(
    "json", 
    data_files="reasoning_dataset_v2_train.json", 
    split="train"
)
print(f"   ✅ Loaded {len(my_dataset)} HICRA examples")

# === 2. Load Nemotron Math Data (Streaming) ===
print(f"🌊 Streaming {NEMOTRON_SAMPLE_SIZE} Nemotron math examples...")
try:
    nemotron_stream = load_dataset(
        "nvidia/Nemotron-Post-Training-Dataset-v1", 
        split="math", 
        streaming=True
    )
    
    # Take a sample and convert to list
    nemotron_list = []
    for i, example in enumerate(nemotron_stream):
        if i >= NEMOTRON_SAMPLE_SIZE:
            break
        formatted = format_nemotron(example)
        # Only keep if it's not too long
        if filter_by_length(formatted):
            nemotron_list.append(formatted)
        
        if (i + 1) % 500 == 0:
            print(f"   Processed {i + 1} examples, kept {len(nemotron_list)}...")
    
    nemotron_dataset = Dataset.from_list(nemotron_list)
    print(f"   ✅ Loaded {len(nemotron_dataset)} Nemotron examples (after length filter)")
    
except Exception as e:
    print(f"   ⚠️ Could not load Nemotron: {e}")
    print("   Continuing with HICRA data only...")
    nemotron_dataset = None

# === 3. Combine Datasets ===
print("🔀 Combining datasets...")

# Filter HICRA by length too
my_dataset_filtered = my_dataset.filter(filter_by_length)
print(f"   HICRA after filter: {len(my_dataset_filtered)} examples")

if nemotron_dataset and len(nemotron_dataset) > 0:
    from datasets import concatenate_datasets
    
    # Make sure both have the same columns
    combined_dataset = concatenate_datasets([my_dataset_filtered, nemotron_dataset])
    print(f"   ✅ Combined dataset: {len(combined_dataset)} examples")
else:
    combined_dataset = my_dataset_filtered
    print(f"   ✅ Using HICRA only: {len(combined_dataset)} examples")

# === 4. Format for GRPO Training ===
print("📝 Formatting for GRPO...")
dataset_train = combined_dataset.map(format_prompt)

# Shuffle to mix the datasets
dataset_train = dataset_train.shuffle(seed=42)

# === 5. Load Test Set (HICRA only) ===
dataset_test = load_dataset(
    "json", 
    data_files="reasoning_dataset_v2_test.json", 
    split="train"
).map(format_prompt)

print(f"\n✅ Final Training Set: {len(dataset_train)} examples")
print(f"✅ Test Set: {len(dataset_test)} examples")
print(f"\nSample prompt format:")
print(dataset_train[0]['prompt'])

# This is the improved HICRA that we need to merge with the Emergent Temporaral - MetaController

In [ ]:
# Cell 10: Original HICRA Reward Functions
# ORIGINAL - Better version of HICRA to cell 8
import re

# Strategic reasoning phrases (from HICRA paper)

STRATEGIC_GRAMS = [
    # Beginning a thought
    "let's analyze", "first we need", "to solve this", "let's assume",
    
    # Logic Connectors (The most important ones)
    "implies that", "consequently", "therefore", "thus", "because", 
    "since", "given that", "conversely", "alternatively",
    
    # Process Checks (Metacognition)
    "checking the", "verifying", "double check", "but wait", "identifying",
    "notice that", "recall that", "we can conclude",
    
    # Mathematical Actions
    "substituting", "calculating", "simplifying", "solving for", "derivative of"
]

def extract_xml_answer(text: str) -> str:
    """Extract answer from <answer> tags."""
    if "<answer>" not in text:
        return text.strip()
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """
    Check if the model's answer matches the expected answer.
    Returns 2.0 for correct, 0.0 for incorrect.
    """
    responses = [completion[0]['content'] for completion in completions]
    extracted = [extract_xml_answer(r) for r in responses]
    
    # Debug output (first item only)
    q = prompts[0][-1]['content'][:100]  # First 100 chars of question
    print(f"---\nQ: {q}...\nExpected: {answer[0]}\nExtracted: {extracted[0][:50]}...")
    
    rewards = []
    for ext, ans in zip(extracted, answer):
        # Check if answer appears in extracted text
        if str(ans).strip() in ext:
            rewards.append(2.0)
        else:
            rewards.append(0.0)
    return rewards

def reasoning_reward_func(completions, **kwargs) -> list[float]:
    """
    HICRA-inspired reward for reasoning structure.
    Gives bonus for using strategic reasoning phrases.
    """
    responses = [completion[0]['content'] for completion in completions]
    rewards = []
    
    for response in responses:
        score = 0.0
        response_lower = response.lower()
        
        # Check for strategic grams
        for gram in STRATEGIC_GRAMS:
            if gram in response_lower:
                score += 0.05
        
        # Bonus for using reasoning tags
        if "<think>" in response and "</think>" in response:
            score += 0.2
        if "<answer>" in response and "</answer>" in response:
            score += 0.1
        
        # Cap the reward
        rewards.append(min(score, 0.5))
    
    return rewards

def format_reward_func(completions, **kwargs) -> list[float]:
    """
    Reward for correct XML format AND stopping correctly.
    """
    rewards = []
    for completion in completions:
        response = completion[0]['content']
        
        # 1. Check if it has the tags
        has_tags = "<think>" in response and "</think>" in response and "<answer>" in response and "</answer>" in response
        
        # 2. Check if it rambles after the answer
        # We split by </answer> and check if there is significant text afterwards
        parts = response.split("</answer>")
        clean_stop = False
        if len(parts) > 1:
            # If the stuff after </answer> is just whitespace or EOS, it's good.
            # If it's another <think> block, it's bad.
            remainder = parts[1].strip()
            if len(remainder) < 5: # Tolerance for tiny noise
                clean_stop = True
        
        score = 0.0
        if has_tags:
            score += 0.5
        if clean_stop:
            score += 0.5 # Big bonus for stopping!
            
        rewards.append(score)
    return rewards

print("✅ Reward functions defined")

In [ ]:
# Cell 11 (updated)
# ORIGINAL

from trl import GRPOConfig, GRPOTrainer

# --- 2. Training Config for RTX 4090 ---
# Explicitly define these variables to avoid NameError
max_prompt_length = 512
max_completion_length = 1024  # 1024 tokens for reasoning

training_args = GRPOConfig(
    output_dir="phi-3.5-hicra-reasoner",
    
    # OPTIMIZATION
    learning_rate=5e-6, # Keep low for stability
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    
    # MEMORY & BATCHING
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1, # Keep small when num_generations is high (16x)
    
    # GRPO SPECIFIC (The "Luxury" Settings)
    num_generations=16,       # <--- 16 samples per question for better variance
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    
    # DURATION
    max_steps=1250, # Start small to test
    save_steps=100,
    logging_steps=1,
    
    # EFFICIENCY
    fp16=False,
    bf16=True, # 4090 loves Bfloat16
    report_to="tensorboard"
)

print(f"✅ Training configuration set")
print(f"   Prompt: {max_prompt_length} tokens, Completion: {max_completion_length} tokens")

In [ ]:
# Cell 12: Initialize Trainer
# ORIGINAL

print("🚀 Initializing GRPO Trainer...")

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        correctness_reward_func,
        reasoning_reward_func,
        format_reward_func,
    ],
    args=training_args,
    train_dataset=dataset_train,
)

print("✅ Trainer initialized!")

In [ ]:
# Cell 13: Run Training!
# ORIGINAL
print("🏋️ Starting training...")
print("Note: First ~100 steps may show 0 reward. Be patient!")
print("="*50)

trainer_stats = trainer.train()

print("="*50)
print("✅ Training complete!")